# Slab DFT: H2 Adsorption on Mg2Ni(0001) with Nb2O5/Fe Catalysts

Three systems compared via adsorption energy:
1. Mg2Ni(0001) + H (baseline)
2. Mg2Ni(0001) + Nb2O5 + H (catalyst)
3. Mg2Ni(0001) + Nb2O5 + Fe + H (co-catalyst)

**Two-pass workflow:**
- Phase 1: Generate Calcs 0-3 (no dependencies)
- Run `pw.x -in nb2o5_cluster.pwi > nb2o5_cluster.pwo` to optimize the cluster
- Phase 2: Generate Calcs 4-7 (reads optimized Nb2O5 geometry)

In [1]:
from ase.io import read, write
from ase import Atoms
from ase.constraints import FixAtoms
import numpy as np

pseudo_dir = '/home/x/Workspace/espresso/pseudo'

all_pseudopotentials = {
    'Mg': 'Mg.pbe-n-kjpaw_psl.0.3.0.UPF',
    'Ni': 'ni_pbe_v1.4.uspp.F.UPF',
    'H':  'H.pbe-rrkjus_psl.1.0.0.UPF',
    'Nb': 'Nb.pbe-spn-kjpaw_psl.0.3.0.UPF',
    'O':  'O.pbe-n-kjpaw_psl.0.1.UPF',
    'Fe': 'Fe.pbe-spn-kjpaw_psl.0.2.1.UPF',
}

# starting_magnetization values per species (ASE reads these from initial_magnetic_moments)
MAGNETIC_MOMENTS = {'Ni': 0.5, 'Fe': 0.5, 'Nb': 0.5}

In [2]:
def set_magnetic_moments(atoms):
    """Set initial_magnetic_moments on atoms based on element type.

    ASE's espresso writer converts per-atom moments to per-species
    starting_magnetization values in the QE input file.
    """
    moments = [MAGNETIC_MOMENTS.get(sym, 0.0) for sym in atoms.get_chemical_symbols()]
    atoms.set_initial_magnetic_moments(moments)

def make_slab_input_params(prefix, species_list):
    """Generate QE input parameters for slab calculations with dipole correction.

    Args:
        prefix: QE calculation prefix
        species_list: ordered list of unique element symbols
    Returns:
        input_params dict, pseudopotentials dict
    """
    input_params = {
        'control': {
            'calculation': 'relax',
            'prefix': prefix,
            'outdir': './out/',
            'pseudo_dir': pseudo_dir,
            'tprnfor': True,
            'tstress': False,
            'etot_conv_thr': 1.4e-4,
            'forc_conv_thr': 1.0e-4,
            'max_seconds': 86000,
            'restart_mode': 'from_scratch',
            'dipfield': True,
            'tefield': True,
        },
        'system': {
            'ecutwfc': 60.0,
            'ecutrho': 800.0,
            'occupations': 'smearing',
            'smearing': 'cold',
            'degauss': 0.01,
            'nspin': 2,
            'edir': 3,
            'emaxpos': 0.95,
            'eopreg': 0.05,
        },
        'electrons': {
            'conv_thr': 1.2e-9,
            'mixing_beta': 0.2,
            'electron_maxstep': 200,
        },
    }

    pseudopotentials = {s: all_pseudopotentials[s] for s in species_list}
    return input_params, pseudopotentials

In [3]:
SLAB_VACUUM = 12  # Angstrom per side (24 A total), clearance for adsorbates + dipole correction

def build_mg2ni_slab():
    """Build Mg2Ni(0001) 2x2x2 slab with vacuum and bottom-half fixed.

    Returns:
        slab: ASE Atoms (144 atoms: 96 Mg + 48 Ni), with magnetic moments set
        z_mid: midpoint z-coordinate for constraint reference
    """
    bulk_struct = read('../mgh2-cif/cif/mg2ni-P62.cif')
    slab = bulk_struct * (2, 2, 2)  # 144 atoms

    slab.center(vacuum=SLAB_VACUUM, axis=2)

    z = slab.positions[:, 2]
    z_mid = (z.min() + z.max()) / 2
    slab.set_constraint(FixAtoms(mask=(z < z_mid)))
    set_magnetic_moments(slab)

    print(f"  Slab: {len(slab)} atoms, cell c = {slab.cell[2,2]:.1f} A")
    print(f"  Z range: {z.min():.1f} - {z.max():.1f} A, fixed below z = {z_mid:.1f} A")
    print(f"  Dipole sawtooth region: {0.90*slab.cell[2,2]:.1f} - {slab.cell[2,2]:.1f} A")

    return slab, z_mid

## Phase 1: Calcs 0-3 (no cluster dependency)

### Calc 0: Isolated Nb2O5 cluster

In [4]:
# Nb2O5 edge-sharing NbO6 dimer: 2 Nb + 5 O
# Bottom bridging O faces slab in later calculations
nb2o5 = Atoms(
    symbols=['Nb', 'Nb', 'O', 'O', 'O', 'O', 'O'],
    positions=[
        [ 0.00,  0.00,  0.00],   # Nb1
        [ 3.30,  0.00,  0.00],   # Nb2
        [ 1.65,  1.00,  0.00],   # O bridging (top)
        [ 1.65, -1.00,  0.00],   # O bridging (side)
        [ 1.65,  0.00, -1.40],   # O bridging (bottom - faces slab later)
        [-1.30,  0.00,  1.20],   # O terminal (Nb1)
        [ 4.60,  0.00,  1.20],   # O terminal (Nb2)
    ],
    cell=[12.0, 12.0, 12.0],
    pbc=True,
)
nb2o5.center()
set_magnetic_moments(nb2o5)

input_params = {
    'control': {
        'calculation': 'relax',
        'prefix': 'nb2o5_cluster',
        'outdir': './out/',
        'pseudo_dir': pseudo_dir,
        'tprnfor': True,
        'tstress': False,
        'etot_conv_thr': 1.4e-4,
        'forc_conv_thr': 1.0e-4,
        'max_seconds': 86000,
        'restart_mode': 'from_scratch',
    },
    'system': {
        'ecutwfc': 60.0,
        'ecutrho': 800.0,
        'occupations': 'smearing',
        'smearing': 'cold',
        'degauss': 0.01,
        'nspin': 2,
    },
    'electrons': {
        'conv_thr': 1.2e-9,
        'mixing_beta': 0.2,
        'electron_maxstep': 200,
    },
}

pseudopotentials = {
    'Nb': all_pseudopotentials['Nb'],
    'O':  all_pseudopotentials['O'],
}

write('nb2o5_cluster.pwi', nb2o5, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=None)
print("Written: nb2o5_cluster.pwi (7 atoms, gamma-point)")

Written: nb2o5_cluster.pwi (7 atoms, gamma-point)


### Calc 1: H2 gas molecule

In [6]:
d = 0.7414  # H-H bond length (Angstrom)
h2 = Atoms('H2', positions=[[0, 0, 0], [0, 0, d]], cell=[10, 10, 10], pbc=True)
h2.center()

input_params = {
    'control': {
        'calculation': 'relax',
        'prefix': 'h2',
        'outdir': './out/',
        'pseudo_dir': pseudo_dir,
        'tprnfor': True,
        'tstress': True,
        'etot_conv_thr': 1.4e-4,
        'forc_conv_thr': 1.0e-4,
        'max_seconds': 27360,
        'restart_mode': 'from_scratch',
    },
    'system': {
        'ecutwfc': 60.0,
        'ecutrho': 800.0,
        'occupations': 'fixed',
    },
    'electrons': {
        'conv_thr': 1.2e-9,
    },
}

pseudopotentials = {'H': all_pseudopotentials['H']}

write('h2.pwi', h2, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=None)
print("Written: h2.pwi (2 atoms, gamma-point)")

Written: h2.pwi (2 atoms, gamma-point)


### Calc 2: Clean Mg2Ni(0001) slab

In [7]:
slab_clean, z_mid_clean = build_mg2ni_slab()

input_params, pseudopotentials = make_slab_input_params('slab_clean', ['Mg', 'Ni'])

write('slab_clean.pwi', slab_clean, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=(4, 4, 1))
print("Written: slab_clean.pwi")

  Slab: 144 atoms, cell c = 49.7 A
  Z range: 12.0 - 37.7 A, fixed below z = 24.9 A
  Dipole sawtooth region: 44.8 - 49.7 A
Written: slab_clean.pwi


### Calc 3: Slab + H

In [8]:
slab_h, z_mid_h = build_mg2ni_slab()

# Find top surface: atoms within 1.5 A of the highest z
z = slab_h.positions[:, 2]
z_top = z.max()
top_mask = z > (z_top - 1.5)
top_positions = slab_h.positions[top_mask]

# Place H at hollow site (centroid of 3 top-layer atoms), 1.6 A above surface
hollow_xy = top_positions[:3, :2].mean(axis=0)
h_pos = [hollow_xy[0], hollow_xy[1], z_top + 1.6]

n_slab = len(slab_h)
slab_h += Atoms('H', positions=[h_pos])
set_magnetic_moments(slab_h)

# Re-apply constraints: only original bottom-half slab atoms are fixed
fixed_indices = [i for i in range(n_slab) if slab_h.positions[i, 2] < z_mid_h]
slab_h.set_constraint(FixAtoms(indices=fixed_indices))

input_params, pseudopotentials = make_slab_input_params('slab_H', ['Mg', 'Ni', 'H'])

write('slab_H.pwi', slab_h, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=(4, 4, 1))
print(f"Written: slab_H.pwi ({len(slab_h)} atoms, H at z = {h_pos[2]:.2f} A)")

  Slab: 144 atoms, cell c = 49.7 A
  Z range: 12.0 - 37.7 A, fixed below z = 24.9 A
  Dipole sawtooth region: 44.8 - 49.7 A
Written: slab_H.pwi (145 atoms, H at z = 39.34 A)


## Phase 2: Calcs 4-7 (requires optimized Nb2O5 from Calc 0)

**Before running these cells:** execute `pw.x -in nb2o5_cluster.pwi > nb2o5_cluster.pwo`

### Calc 4: Slab + Nb2O5

In [9]:
# Read optimized Nb2O5 cluster
cluster = read('nb2o5_cluster.pwo')
print(f"Optimized Nb2O5: {len(cluster)} atoms")

slab, z_mid = build_mg2ni_slab()
n_slab = len(slab)
z_top = slab.positions[:, 2].max()

# Place cluster: bottom atom 2.2 A above slab surface, centered laterally
cluster_z = cluster.positions[:, 2]
cluster.positions[:, 2] += (z_top + 2.2 - cluster_z.min())

slab_center_xy = (slab.cell[0, :2] + slab.cell[1, :2]) / 2
cluster_center_xy = cluster.positions[:, :2].mean(axis=0)
xy_shift = slab_center_xy - cluster_center_xy
cluster.positions[:, 0] += xy_shift[0]
cluster.positions[:, 1] += xy_shift[1]

slab_nb2o5 = slab + cluster
set_magnetic_moments(slab_nb2o5)

# Fix bottom-half slab atoms only
fixed = [i for i in range(n_slab) if slab_nb2o5.positions[i, 2] < z_mid]
slab_nb2o5.set_constraint(FixAtoms(indices=fixed))

input_params, pseudopotentials = make_slab_input_params('slab_Nb2O5', ['Mg', 'Ni', 'Nb', 'O'])

write('slab_Nb2O5.pwi', slab_nb2o5, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=(4, 4, 1))

z_max_atom = slab_nb2o5.positions[:, 2].max()
z_sawtooth = 0.90 * slab_nb2o5.cell[2, 2]
print(f"Written: slab_Nb2O5.pwi ({len(slab_nb2o5)} atoms)")
print(f"  Highest atom z = {z_max_atom:.2f} A, sawtooth starts at {z_sawtooth:.2f} A")
if z_max_atom > z_sawtooth:
    print("  WARNING: atoms overlap with dipole correction region! Increase SLAB_VACUUM.")

Optimized Nb2O5: 7 atoms
  Slab: 144 atoms, cell c = 49.7 A
  Z range: 12.0 - 37.7 A, fixed below z = 24.9 A
  Dipole sawtooth region: 44.8 - 49.7 A
Written: slab_Nb2O5.pwi (151 atoms)
  Highest atom z = 42.23 A, sawtooth starts at 44.76 A


### Calc 5: Slab + Nb2O5 + H

In [10]:
slab_nb2o5_h = slab_nb2o5.copy()
n_slab = 144  # number of slab atoms

# Place H at the interface between cluster and slab surface
z_slab_top = slab_nb2o5_h.positions[:n_slab, 2].max()
cluster_pos = slab_nb2o5_h.positions[n_slab:]
cluster_center_xy = cluster_pos[:, :2].mean(axis=0)

# H at slab surface + 1.6 A, slightly offset from cluster center
h_pos = [cluster_center_xy[0] + 1.5, cluster_center_xy[1], z_slab_top + 1.6]

slab_nb2o5_h += Atoms('H', positions=[h_pos])
set_magnetic_moments(slab_nb2o5_h)

# Re-apply constraints
fixed = [i for i in range(n_slab) if slab_nb2o5_h.positions[i, 2] < z_mid]
slab_nb2o5_h.set_constraint(FixAtoms(indices=fixed))

input_params, pseudopotentials = make_slab_input_params('slab_Nb2O5_H', ['Mg', 'Ni', 'Nb', 'O', 'H'])

write('slab_Nb2O5_H.pwi', slab_nb2o5_h, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=(4, 4, 1))
print(f"Written: slab_Nb2O5_H.pwi ({len(slab_nb2o5_h)} atoms, H at z = {h_pos[2]:.2f} A)")

Written: slab_Nb2O5_H.pwi (152 atoms, H at z = 39.34 A)


### Calc 6: Slab + Nb2O5 + Fe

In [11]:
slab_nb2o5_fe = slab_nb2o5.copy()
n_slab = 144

# Place Fe ~2.5 A from first Nb atom, on the surface near the cluster
nb1_pos = slab_nb2o5_fe.positions[n_slab]  # first Nb atom
z_slab_top = slab_nb2o5_fe.positions[:n_slab, 2].max()

fe_pos = [nb1_pos[0] + 2.0, nb1_pos[1] + 1.5, z_slab_top + 2.0]
fe_nb_dist = np.linalg.norm(np.array(fe_pos) - nb1_pos)

slab_nb2o5_fe += Atoms('Fe', positions=[fe_pos])
set_magnetic_moments(slab_nb2o5_fe)

# Re-apply constraints
fixed = [i for i in range(n_slab) if slab_nb2o5_fe.positions[i, 2] < z_mid]
slab_nb2o5_fe.set_constraint(FixAtoms(indices=fixed))

input_params, pseudopotentials = make_slab_input_params('slab_Nb2O5_Fe', ['Mg', 'Ni', 'Nb', 'O', 'Fe'])

write('slab_Nb2O5_Fe.pwi', slab_nb2o5_fe, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=(4, 4, 1))
print(f"Written: slab_Nb2O5_Fe.pwi ({len(slab_nb2o5_fe)} atoms)")
print(f"  Fe at z = {fe_pos[2]:.2f} A, Fe-Nb distance = {fe_nb_dist:.2f} A")

Written: slab_Nb2O5_Fe.pwi (152 atoms)
  Fe at z = 39.74 A, Fe-Nb distance = 3.04 A


### Calc 7: Slab + Nb2O5 + Fe + H

In [12]:
slab_nb2o5_fe_h = slab_nb2o5_fe.copy()
n_slab = 144

# Place H near the Fe atom at the interface
fe_pos_current = slab_nb2o5_fe_h.positions[-1]  # last atom is Fe
z_slab_top = slab_nb2o5_fe_h.positions[:n_slab, 2].max()

# H between Fe and the slab surface
h_pos = [fe_pos_current[0] + 0.5, fe_pos_current[1] - 0.5, z_slab_top + 1.6]

slab_nb2o5_fe_h += Atoms('H', positions=[h_pos])
set_magnetic_moments(slab_nb2o5_fe_h)

# Re-apply constraints
fixed = [i for i in range(n_slab) if slab_nb2o5_fe_h.positions[i, 2] < z_mid]
slab_nb2o5_fe_h.set_constraint(FixAtoms(indices=fixed))

input_params, pseudopotentials = make_slab_input_params('slab_Nb2O5_Fe_H', ['Mg', 'Ni', 'Nb', 'O', 'Fe', 'H'])

write('slab_Nb2O5_Fe_H.pwi', slab_nb2o5_fe_h, format='espresso-in',
      input_data=input_params, pseudopotentials=pseudopotentials, kpts=(4, 4, 1))
print(f"Written: slab_Nb2O5_Fe_H.pwi ({len(slab_nb2o5_fe_h)} atoms, H at z = {h_pos[2]:.2f} A)")

# Final verification
print("\n--- Verification ---")
for fname in ['nb2o5_cluster.pwi', 'h2.pwi', 'slab_clean.pwi', 'slab_H.pwi',
              'slab_Nb2O5.pwi', 'slab_Nb2O5_H.pwi', 'slab_Nb2O5_Fe.pwi', 'slab_Nb2O5_Fe_H.pwi']:
    try:
        atoms = read(fname, format='espresso-in')
        z = atoms.positions[:, 2]
        symbols = set(atoms.get_chemical_symbols())
        print(f"{fname:30s}  {len(atoms):3d} atoms  species: {sorted(symbols)}  z: {z.min():.1f}-{z.max():.1f} A")
    except FileNotFoundError:
        print(f"{fname:30s}  (Phase 2 - generate after Calc 0 runs)")

Written: slab_Nb2O5_Fe_H.pwi (153 atoms, H at z = 39.34 A)

--- Verification ---
nb2o5_cluster.pwi                 7 atoms  species: ['Nb', 'O']  z: 4.7-7.3 A
h2.pwi                            2 atoms  species: ['H']  z: 4.6-5.4 A
slab_clean.pwi                  144 atoms  species: ['Mg', 'Ni']  z: 12.0-37.7 A
slab_H.pwi                      145 atoms  species: ['H', 'Mg', 'Ni']  z: 12.0-39.3 A
slab_Nb2O5.pwi                  151 atoms  species: ['Mg', 'Nb', 'Ni', 'O']  z: 12.0-42.2 A
slab_Nb2O5_H.pwi                152 atoms  species: ['H', 'Mg', 'Nb', 'Ni', 'O']  z: 12.0-42.2 A
slab_Nb2O5_Fe.pwi               152 atoms  species: ['Fe', 'Mg', 'Nb', 'Ni', 'O']  z: 12.0-42.2 A
slab_Nb2O5_Fe_H.pwi             153 atoms  species: ['Fe', 'H', 'Mg', 'Nb', 'Ni', 'O']  z: 12.0-42.2 A
